<a href="https://colab.research.google.com/github/shiosabax/test_web_program/blob/shiosabax-patch-1/kabuka_sihyou.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import requests
import json

# 1. 取得したい主要株価指数のシンボル（識別子）を「リスト」で定義
# ^N225: 日経平均株価, ^TOPX: TOPIX, ^GSPC: S&P500, ^IXIC: NASDAQ総合
indices = ["^N225", "^TOPX", "^GSPC", "^IXIC"]

# サーバーにブラウザからのアクセスであることを伝えるヘッダー情報
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("=== 主要株価指数のデータ取得開始 ===")

# 2. リストの中身を for ループで1つずつ順番に処理
for symbol in indices:
    # データを取得するURL（Yahoo Financeの公開エンドポイント形式）
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}?interval=1d&range=1d"

    # requestsを使ってデータを取得
    response = requests.get(url, headers=headers)

    # 通信が成功（ステータスコード 200）したか確認
    if response.status_code == 200:
        data = response.json()

        try:
            meta = data["chart"]["result"][0]["meta"]
            name = meta.get("shortName", symbol)
            currency = meta.get("currency", "")
            current_price = meta.get("regularMarketPrice", 0)
            prev_close = meta.get("chartPreviousClose", current_price)

            # 前日比と変動率の計算
            diff = current_price - prev_close
            diff_percent = (diff / prev_close) * 100 if prev_close else 0

            # 結果を分かりやすく出力
            sign = "+" if diff >= 0 else ""
            print(f"【{name} ({symbol})】")
            print(f"  最新値: {current_price:,.2f} {currency}")
            print(f"  前日比: {sign}{diff:,.2f} ({sign}{diff_percent:.2f}%)\n")

        except (KeyError, IndexError) as e:
            print(f"【{symbol}】データの解析に失敗しました。")
    else:
        print(f"【{symbol}】データの取得に失敗しました（ステータス: {response.status_code}）")

    # サーバー負荷軽減のため1秒待機
    time.sleep(1)

print("=== 取得完了 ===")

=== 主要株価指数のデータ取得開始 ===
【Nikkei 225 (^N225)】
  最新値: 66,405.56 JPY
  前日比: +273.58 (+0.41%)

【^TOPX (^TOPX)】
  最新値: 0.00 None
  前日比: +0.00 (+0.00%)

【S&P 500 (^GSPC)】
  最新値: 7,711.76 USD
  前日比: -19.23 (-0.25%)

【NASDAQ Composite (^IXIC)】
  最新値: 26,402.42 USD
  前日比: -138.93 (-0.52%)

=== 取得完了 ===
